# Phase-2: Feature Engineering per LSOA (England)

**Goal:** Build a master feature matrix one row per England LSOA (~33,755), one column per feature.

**Features computed:**
| Feature | Source |
|---|---|
| `crime_count` | Total crimes per LSOA (36 months) |
| `severity_weighted_count` | CCHI-weighted crime count per LSOA |
| `resolution_rate` | % crimes with positive outcome per LSOA |
| `imd_rank` | IoD 2025 overall deprivation rank |
| `income_rank` | IoD 2025 income deprivation rank |
| `employment_rank` | IoD 2025 employment deprivation rank |

**Note:** stop_search_rate, total_footfall, and seasonal_volatility are not included in this England-wide model.

**Output:** `England/outputs/phase2/phase2_feature_matrix.parquet`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Dataset Loading
BASE     = Path('Dataset path')
P1       = BASE / 'england_final_light' / 'outputs' / 'phase1'
OUT      = BASE / 'england_final_light' / 'outputs' / 'phase2'
OUT.mkdir(parents=True, exist_ok=True)

IMD_FILE = BASE / 'data' / 'IoD-2025-custom_data_download-LSOA.csv'

# Cambridge Crime Harm Index (CCHI 2026) severity weights for police.uk categories.
# Derived in 'severity weight/' from the official CCHI 2026 table: mean CCHI score
# (days of custody at the sentencing starting point) of the offences in each category.
# See severity weight/severity_weight_justification.md for methodology and limitations.
CCHI_WEIGHTS = {
    'Violence and sexual offences': 670,
    'Robbery':                      365,
    'Burglary':                     281,
    'Vehicle crime':                  6,
    'Theft from the person':          2,
    'Shoplifting':                    1,
    'Other theft':                    4,
    'Bicycle theft':                  5,
    'Criminal damage and arson':     98,
    'Drugs':                        156,
    'Public order':                  53,
    'Possession of weapons':        541,
    'Other crime':                   74,
    'Anti-social behaviour':          1,
}

# Positive outcome categories (resolution_rate)
RESOLVED_OUTCOMES = {
    'Suspect charged',
    'Offender given a caution',
    'Offender given a penalty notice',
    'Offender fined',
    'Offender deported',
    'Offender otherwise dealt with',
    'Suspect charged as part of another case',
    'Local resolution',
    'Offender given a drugs possession warning',
    'Offender given conditional discharge',
    'Offender given absolute discharge',
    'Offender sent to prison',
    'Offender given suspended prison sentence',
    'Offender given community sentence',
}

print('Config loaded.')
print('Output folder:', OUT)

## Section-1: Load Data

In [ ]:
print('Loading Phase 1 parquets...')
crimes   = pd.read_parquet(P1 / 'phase1_crimes_england.parquet')
outcomes = pd.read_parquet(P1 / 'phase1_outcomes_england.parquet')

print(f'  crimes:   {crimes.shape}')
print(f'  outcomes: {outcomes.shape}')

In [ ]:
# Build England LSOA backbone from IoD file
# IoD covers all 33,755 England LSOAs and includes LSOA name and LAD name
print('Building England LSOA backbone from IoD 2025...')
imd_raw  = pd.read_csv(IMD_FILE)
backbone = imd_raw[[
    'LSOA code (2021)',
    'LSOA name (2021)',
    'Local Authority District name (2024)',
]].copy()
backbone.columns = ['lsoa21cd', 'lsoa21nm', 'lad22nm']

ENGLAND_LSOAS = set(backbone['lsoa21cd'])
print(f'  Total England LSOAs: {len(backbone)}')
print(f'  Unique LADs: {backbone["lad22nm"].nunique()}')

## Section-2: Filter Crimes to England LSOAs
The crime data may contain records with LSOA codes outside England (e.g. Wales, or missing codes). Filter to LSOAs present in the IoD backbone.

In [ ]:
before = len(crimes)
crimes_england = crimes[crimes['LSOA code'].isin(ENGLAND_LSOAS)].copy()
crimes_england = crimes_england.rename(columns={'LSOA code': 'lsoa21cd'})
print(f'Crimes before filter: {before:,}')
print(f'Crimes after filter:  {len(crimes_england):,}  ({before - len(crimes_england):,} outside England dropped)')
print(f'Unique England LSOAs in crime data: {crimes_england["lsoa21cd"].nunique()}')

## Section-3: Crime Count & Severity-Weighted Count

In [ ]:
# Apply CCHI weights
crimes_england['cchi_weight'] = crimes_england['Crime type'].map(CCHI_WEIGHTS).fillna(74)  # default=Other crime (CCHI 2026 mean)

crime_features = (
    crimes_england
    .groupby('lsoa21cd')
    .agg(
        crime_count=('Crime ID', 'count'),
        severity_weighted_count=('cchi_weight', 'sum'),
    )
    .reset_index()
)

print(f'Shape: {crime_features.shape}')
print(crime_features.describe().round(1))
crime_features.head(3)

## Section-4: Resolution Rate per LSOA

In [ ]:
# Join outcomes to crimes on Crime ID
crimes_with_id  = crimes_england[crimes_england['Crime ID'].notna()][['Crime ID', 'lsoa21cd']].copy()
outcomes_clean  = outcomes[outcomes['Crime ID'].notna()][['Crime ID', 'Outcome type']].copy()

merged = crimes_with_id.merge(outcomes_clean, on='Crime ID', how='left')
merged['resolved'] = merged['Outcome type'].apply(
    lambda x: any(r.lower() in str(x).lower() for r in RESOLVED_OUTCOMES) if pd.notna(x) else False
)

resolution = (
    merged
    .groupby('lsoa21cd')
    .agg(
        crimes_with_outcome=('Crime ID', 'count'),
        resolved_count=('resolved', 'sum'),
    )
    .reset_index()
)
resolution['resolution_rate'] = (
    resolution['resolved_count'] / resolution['crimes_with_outcome'] * 100
).round(2)

resolution = resolution[['lsoa21cd', 'resolution_rate']]
print(f'Shape: {resolution.shape}')
print(resolution['resolution_rate'].describe().round(2))
resolution.head(3)

## Section-5: IMD Deprivation Ranks

In [ ]:
print('Loading IMD 2025...')
imd = imd_raw[[
    'LSOA code (2021)',
    'Index of Multiple Deprivation (IMD) Rank',
    'Income Rank',
    'Employment Rank',
]].copy()
imd.columns = ['lsoa21cd', 'imd_rank', 'income_rank', 'employment_rank']
print(f'  IMD shape: {imd.shape}')
print(f'  Nulls: {imd.isnull().sum().to_dict()}')

## Section-6: Assemble Master Feature Matrix

In [ ]:
# Start from backbone (all 33,755 England LSOAs from IoD)
feature_matrix = backbone.copy()

# Join all features
feature_matrix = feature_matrix.merge(crime_features, on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(resolution,     on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(imd,            on='lsoa21cd', how='left')

# Fill nulls with 0 for count-based features (LSOAs with no events)
fill_zero = ['crime_count', 'severity_weighted_count']
feature_matrix[fill_zero] = feature_matrix[fill_zero].fillna(0)

# resolution_rate: LSOAs with no crimes get NaN (can't compute rate) — leave as NaN

print(f'Shape: {feature_matrix.shape}')
print(f'\nNull counts:')
print(feature_matrix.isnull().sum())
feature_matrix.head(3)

In [ ]:
# Descriptive summary
numeric_cols = ['crime_count', 'severity_weighted_count', 'resolution_rate',
                'imd_rank', 'income_rank', 'employment_rank']
print(feature_matrix[numeric_cols].describe().round(2).to_string())

## Section - 7: Save Outputs

In [ ]:
# Save feature matrix
feature_matrix.to_parquet(OUT / 'phase2_feature_matrix.parquet', index=False)
print(f'Saved phase2_feature_matrix.parquet — {feature_matrix.shape}')

# Save monthly crime counts per LSOA (used in Phase 3 for Kruskal-Wallis)
all_months = crimes_england['Month'].sort_values().unique()
all_lsoas  = crimes_england['lsoa21cd'].unique()

monthly_counts = (
    crimes_england
    .groupby(['lsoa21cd', 'Month'])
    .size()
    .reset_index(name='monthly_crime_count')
)

full_index = pd.MultiIndex.from_product([all_lsoas, all_months], names=['lsoa21cd', 'Month'])
monthly_counts = (
    monthly_counts
    .set_index(['lsoa21cd', 'Month'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

monthly_counts.to_parquet(OUT / 'phase2_monthly_crime_counts.parquet', index=False)
print(f'Saved phase2_monthly_crime_counts.parquet — {monthly_counts.shape}')

print('\nPhase 2 complete.')

In [ ]:
# Phase-2
print('=' * 55)
print('PHASE 2 SUMMARY (England)')
print('=' * 55)
print(f'Total LSOAs in matrix:          {len(feature_matrix):>6}')
print(f'LSOAs with crime data:          {(feature_matrix["crime_count"] > 0).sum():>6}')
print(f'LSOAs with IMD data:            {feature_matrix["imd_rank"].notna().sum():>6}')
print(f'LSOAs with resolution rate:     {feature_matrix["resolution_rate"].notna().sum():>6}')
print('=' * 55)
print(f'Outputs saved to: {OUT}')